# Phishing Cross-Dataset Feature Robustness Study — Preprocessing
**Pipeline**
1. Load datasets
2. Standardize column names
3. Parsing some url features for B
3. Map raw names → unified schema (URL + HTML)
4. Fix semantics and derive missing overlap features
5. Drop bad columns
6. Impute missing values
7. Cap outliers
8. Scale per dataset
9. Extract overlap feature space
10. Save cleaned outputs

## 0. Setup


In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', 250)
RANDOM_SEED = 42


## 1. Load datasets


In [ ]:
DATA_A_PATH = "DatasetA.csv"
DATA_B_PATH = "DatasetB.csv"
DATA_C_PATH = "DatasetC.csv"

df_A = pd.read_csv(DATA_A_PATH)
df_B = pd.read_csv(DATA_B_PATH)
df_C = pd.read_csv(DATA_C_PATH)

print(df_A.shape, df_B.shape, df_C.shape)
display(df_A.head())


(10000, 50) (11430, 89) (79987, 82)


,id,NumDots,SubdomainLevel,PathLevel,UrlLength,NumDash,NumDashInHostname,AtSymbol,TildeSymbol,NumUnderscore,NumPercent,NumQueryComponents,NumAmpersand,NumHash,NumNumericChars,NoHttps,RandomString,IpAddress,DomainInSubdomains,DomainInPaths,HttpsInHostname,HostnameLength,PathLength,QueryLength,DoubleSlashInPath,NumSensitiveWords,EmbeddedBrandName,PctExtHyperlinks,PctExtResourceUrls,ExtFavicon,InsecureForms,RelativeFormAction,ExtFormAction,AbnormalFormAction,PctNullSelfRedirectHyperlinks,FrequentDomainNameMismatch,FakeLinkInStatusBar,RightClickDisabled,PopUpWindow,SubmitInfoToEmail,IframeOrFrame,MissingTitle,ImagesOnlyInForm,SubdomainLevelRT,UrlLengthRT,PctExtResourceUrlsRT,AbnormalExtFormActionR,ExtMetaScriptLinkRT,PctExtNullSelfRedirectHyperlinksRT,CLASS_LABEL
0,1,3,1,5,72,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,21,44,0,0,0,0,0.000,0.250000,1,1,0,0,0,0.0,0,0,0,0,0,0,0,1,1,0,1,1,-1,1,1
1,2,3,1,3,144,0,0,0,0,2,0,2,1,0,41,1,0,0,0,0,0,17,16,103,0,1,0,0.000,0.000000,0,1,0,0,0,0.0,0,0,0,0,0,0,0,0,1,-1,1,1,1,1,1
2,3,3,1,2,58,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,27,24,0,0,0,0,0.375,1.000000,1,1,0,0,0,0.0,0,0,0,0,0,0,0,0,1,0,-1,1,-1,0,1
3,4,3,1,6,79,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,22,50,0,0,0,1,1.000,0.095238,1,1,0,0,0,0.0,1,0,0,0,1,0,0,0,1,-1,1,1,1,-1,1
4,5,3,0,4,46,0,0,0,0,0,0,0,0,0,2,1,1,0,0,1,0,10,29,0,0,0,0,1.000,1.000000,0,0,0,1,0,0.0,1,0,0,0,0,1,0,0,1,1,-1,0,-1,-1,1


## 2. Standardize column names


In [ ]:
def standardize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.lower()
        .str.replace('[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )
    return df

df_A = standardize_columns(df_A)
df_B = standardize_columns(df_B)
df_C = standardize_columns(df_C)


## 3A. Dataset B URL parsing (derive lexical overlap features from raw URL)

Dataset B includes a raw `url` column. This cell parses it to derive missing/ambiguous overlap features.
Run this **before mapping**.


In [ ]:
from urllib.parse import urlparse
import ipaddress

def derive_from_url_B(df, url_col='url'):
    """Parse raw URL column in Dataset B and add derived lexical overlap features."""
    df = df.copy()
    if url_col not in df.columns:
        print(f"No '{url_col}' column found; skipping URL parsing.")
        return df

    urls = df[url_col].astype(str)

    def safe_parse(u):
        try:
            return urlparse(u)
        except Exception:
            return urlparse('')

    parsed = urls.apply(safe_parse)
    scheme = parsed.apply(lambda p: p.scheme.lower() if p.scheme else '')
    netloc = parsed.apply(lambda p: p.netloc.lower() if p.netloc else '')
    path = parsed.apply(lambda p: p.path if p.path else '')
    query = parsed.apply(lambda p: p.query if p.query else '')

    # Strip credentials/port from hostname
    host = netloc.str.replace(r'^.*@', '', regex=True)
    host_no_port = host.str.replace(r':\d+$', '', regex=True)

    # Derived unified overlap features
    df['query_length'] = query.str.len()
    df['query_param_count'] = query.apply(lambda q: 0 if q=='' else q.count('&') + 1)
    df['path_length'] = path.str.len()
    df['path_depth'] = path.apply(lambda p: len([seg for seg in p.split('/') if seg]))

    df['num_at_symbols'] = urls.str.count('@')
    df['num_percents'] = urls.str.count('%')
    df['num_dots'] = host_no_port.str.count('\.')
    df['num_hyphens'] = host_no_port.str.count('-') + path.str.count('-')
    df['num_numeric_chars'] = urls.str.count(r'\d')

    def is_ip(h):
        try:
            ipaddress.ip_address(h)
            return 1
        except Exception:
            return 0
    df['has_ip_address'] = host_no_port.apply(is_ip)

    if 'subdomain_depth' not in df.columns:
        df['subdomain_depth'] = host_no_port.apply(lambda h: max(0, len(h.split('.')) - 2) if h else 0)

    return df

df_B = derive_from_url_B(df_B, url_col='url')
display(df_B[[c for c in ['url','query_length','path_length','path_depth','num_dots','num_hyphens','num_numeric_chars','uses_https','has_ip_address'] if c in df_B.columns]].head())


<>:37: SyntaxWarning: invalid escape sequence '\.'
<>:37: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-590812177.py:37: SyntaxWarning: invalid escape sequence '\.'
  df['num_dots'] = host_no_port.str.count('\.')


,url,query_length,path_length,path_depth,num_dots,num_hyphens,num_numeric_chars,has_ip_address
0,http://www.crestonwood.com/router.php,0,11,1,2,0,0,0
1,http://shadetreetechnology.com/V4/validation/a...,0,47,3,1,0,17,0
2,https://support-appleld.com.secureupdate.duila...,47,20,2,4,1,19,0
3,http://rgipt.ac.in,0,0,0,2,0,0,0
4,http://www.iracing.com/tracks/gateway-motorspo...,0,33,2,2,2,0,0


## 3. Unified mapping (URL + HTML / JS / content)



In [ ]:
COLUMN_MAPPING = {
    # ---- URL / host lexical lengths ----
    'url_length': ['urllength', 'length_url', 'url_length'],
    'hostname_length': ['hostnamelength', 'length_hostname', 'domain_length', 'hostname_length'],
    'path_length': ['pathlength', 'path_length'],
    'query_length': ['querylength', 'query_length'],

    # ---- Lexical counts ----
    'num_dots': ['numdots', 'nb_dots', 'num_dots'],
    'num_hyphens': ['numdash', 'numdashinhostname', 'nb_hyphens', 'num_hyphens'],
    'num_at_symbols': ['atsymbol', 'nb_at', 'num_at'],
    'num_percents': ['numpercent', 'nb_percent', 'num_pct'],
    'num_underscores': ['numunderscore', 'nb_underscore', 'num_underscores'],
    'num_ampersands': ['numampersand', 'nb_and', 'num_ampersands'],
    'num_numeric_chars': ['numnumericchars', 'num_digits', 'ratio_digits_url'],

    # ---- Subdomain / path depth ----
    'subdomain_depth': ['subdomainlevel', 'nb_subdomains', 'subdomain_depth'],
    'path_depth': ['pathlevel', 'path_depth', 'nb_slash'],

    # ---- IP / protocol ----
    'has_ip_address': ['ipaddress', 'ip', 'has_ip_host', 'domain_has_ip'],
    'uses_https': ['nohttps', 'https_token', 'uses_https'],

    # ---- Forms / authentication ----
    'has_insecure_forms': ['insecureforms'],
    'form_action_external': ['extformaction', 'sfh', 'form_action_external'],
    'relative_form_action': ['relativeformaction', 'sfh'],
    'submit_email': ['submitinfotoemail', 'submit_email'],
    'has_login_keywords': ['login_form', 'has_login_keywords'],

    # ---- Title / favicon ----
    'missing_title': ['missingtitle', 'empty_title'],
    'external_favicon': ['extfavicon', 'external_favicon', 'has_favicon_external'],

    # ---- Iframe / popup / JS behavior ----
    'has_iframe': ['iframeorframe', 'iframe', 'n_iframes'],
    'popup_window': ['popupwindow', 'popup_window'],
    'right_click_disabled': ['rightclickdisabled', 'right_clic'],
    'mouseover_script': ['fakelinkinstatusbar', 'onmouseover'],

    # ---- External resources / links ----
    'ratio_external_links': ['pctexthyperlinks', 'ratio_exthyperlinks', 'ratio_external_links'],
    'ratio_null_links': ['pctnullselfredirecthyperlinks', 'ratio_nullhyperlinks', 'ratio_null_links'],
    'external_css_ratio': ['pctextresourceurls', 'nb_extcss', 'css_external_ratio'],

    # ---- Scripts / images ----
    'num_scripts': ['n_scripts'],
    'external_scripts': ['n_external_scripts'],
    'inline_script_ratio': ['inline_script_ratio'],
    'has_images': ['imagesonlyinform'], # For dataset A imagesonlyinform is narrower

    # ---- Label ----
    'label': ['class_label', 'status', 'label', 'class'],
}


## 4. Apply mapping


In [ ]:
def map_columns(df, mapping):
    df = df.copy()
    rename_dict = {}
    lower_cols = {c.lower(): c for c in df.columns}
    for unified, candidates in mapping.items():
        for cand in candidates:
            if cand.lower() in lower_cols:
                rename_dict[lower_cols[cand.lower()]] = unified
                break
    return df.rename(columns=rename_dict)

df_A = map_columns(df_A, COLUMN_MAPPING)
df_B = map_columns(df_B, COLUMN_MAPPING)
df_C = map_columns(df_C, COLUMN_MAPPING)


## 5. Fix semantics + derive remaining overlap features


In [ ]:
def fix_special_cases_and_derive(df):
    df = df.copy()

    # ---- 1) uses_https inversion for Dataset A ----
    # If raw nohttps still exists, make sure uses_https reflects the inverse.
    if "nohttps" in df.columns and "uses_https" not in df.columns:
        df["uses_https"] = df["nohttps"]

    if "uses_https" in df.columns:
        s = df["uses_https"]
        if isinstance(s, pd.DataFrame):  # duplicate protection
            s = s.iloc[:, 0]
        vals = set(pd.Series(s).dropna().unique())
        if vals.issubset({0, 1}) and "nohttps" in df.columns:
            df["uses_https"] = 1 - s

    # ---- 2) path_depth from nb_slash (Dataset B raw) ----
    if "path_depth" not in df.columns and "nb_slash" in df.columns:
        df["path_depth"] = (df["nb_slash"] - 2).clip(lower=0)

    # ---- 3) missing_title from title_len (Dataset C raw) ----
    if "missing_title" not in df.columns and "title_len" in df.columns:
        df["missing_title"] = (df["title_len"] == 0).astype(int)

    # ---- 4) has_iframe from n_iframes (Dataset C raw) ----
    if "has_iframe" not in df.columns and "n_iframes" in df.columns:
        df["has_iframe"] = (df["n_iframes"] > 0).astype(int)

    # ---- 5) has_images from n_images (Dataset C raw) ----
    if "has_images" not in df.columns and "n_images" in df.columns:
        df["has_images"] = (df["n_images"] > 0).astype(int)

    # ---- 6) submit_email from n_mailto_links (Dataset C raw) ----
    if "submit_email" not in df.columns and "n_mailto_links" in df.columns:
        df["submit_email"] = (df["n_mailto_links"] > 0).astype(int)

    # ---- 7) has_images proxy from media ratios (Dataset B raw) ----
    if "has_images" not in df.columns and (
        "ratio_intmedia" in df.columns or "ratio_extmedia" in df.columns
    ):
        rim = df.get("ratio_intmedia", 0)
        rem = df.get("ratio_extmedia", 0)
        df["has_images"] = ((rim + rem) > 0).astype(int)

    # ---- 8) inline_script_ratio correctly ----
    # After mapping, inline_script_ratio may be a COUNT (from n_inline_scripts).
    inline_count = df.get("inline_script_ratio", df.get("n_inline_scripts", None))
    scripts_total = df.get("num_scripts", df.get("n_scripts", None))

    if inline_count is not None and scripts_total is not None:
        denom = scripts_total.replace(0, np.nan)
        df["inline_script_ratio"] = (inline_count / denom).fillna(0.0)

    return df


df_A = fix_special_cases_and_derive(df_A)
df_B = fix_special_cases_and_derive(df_B)
df_C = fix_special_cases_and_derive(df_C)


## 6. Drop bad columns


In [ ]:
def drop_bad_columns(df, missing_thresh=0.5):
    df = df.copy()

    # drop mostly-missing columns
    bad_missing = df.columns[df.isna().mean() > missing_thresh]
    df = df.drop(columns=bad_missing, errors="ignore")

    # remove duplicated columns (critical)
    df = df.loc[:, ~df.columns.duplicated()]

    # keep numeric columns + label if present
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    keep_cols = numeric_cols.copy()

    if "label" in df.columns and "label" not in keep_cols:
        keep_cols.append("label")

    return df[keep_cols]


df_A = drop_bad_columns(df_A)
df_B = drop_bad_columns(df_B)
df_C = drop_bad_columns(df_C)


In [ ]:
if 'id' in df_A.columns:
    df_A = df_A.drop(columns=['id'])

if 'rec_id' in df_C.columns:
    df_C = df_C.drop(columns=['rec_id'])

print(df_A.shape, df_B.shape, df_C.shape)

(10000, 49) (11430, 93) (79987, 75)


## 7. Impute missing values


In [ ]:
def impute_missing(df):
    df = df.copy()
    binary_cols = []
    for c in df.columns:
        if c == 'label':
            continue
        uniq = set(df[c].dropna().unique())
        if uniq.issubset({0,1}) and len(uniq) <= 2:
            binary_cols.append(c)
    df[binary_cols] = df[binary_cols].fillna(0)
    num_cols = [c for c in df.select_dtypes(include='number').columns if c != 'label']
    for c in num_cols:
        if c not in binary_cols:
            df[c] = df[c].fillna(df[c].median())
    return df

df_A = impute_missing(df_A)
df_B = impute_missing(df_B)
df_C = impute_missing(df_C)


## 8. Scale per dataset


In [ ]:
def scale_dataset(df, scaler=None):
    df = df.copy()
    num_cols = [c for c in df.select_dtypes(include='number').columns if c != 'label']
    if scaler is None:
        scaler = StandardScaler()
        df[num_cols] = scaler.fit_transform(df[num_cols])
    else:
        df[num_cols] = scaler.transform(df[num_cols])
    return df, scaler

df_A_scaled, scaler_A = scale_dataset(df_A)
df_B_scaled, scaler_B = scale_dataset(df_B)
df_C_scaled, scaler_C = scale_dataset(df_C)


## 9. Extract overlap feature space


In [ ]:
OVERLAP_FEATURES = [
    'url_length', 'hostname_length', 'num_dots', 'num_hyphens',
    'num_at_symbols', 'num_percents', 'num_numeric_chars',
    'subdomain_depth', 'path_length', 'path_depth', 'query_length',
    'has_ip_address', 'uses_https',
    'has_insecure_forms', 'form_action_external', 'relative_form_action', 'submit_email',
    'has_login_keywords', 'missing_title', 'external_favicon', 'has_iframe',
    'ratio_external_links', 'ratio_null_links', 'external_css_ratio',
    'external_scripts', 'inline_script_ratio', 'has_images','popup_window',
]
OVERLAP_FEATURES = [f for f in OVERLAP_FEATURES if f in df_A_scaled.columns and f in df_B_scaled.columns and f in df_C_scaled.columns]
print('# overlap features:', len(OVERLAP_FEATURES))
print(OVERLAP_FEATURES)

df_A_overlap = df_A_scaled[OVERLAP_FEATURES + ['label']]
df_B_overlap = df_B_scaled[OVERLAP_FEATURES + ['label']]
df_C_overlap = df_C_scaled[OVERLAP_FEATURES + ['label']]


# overlap features: 23
['url_length', 'hostname_length', 'num_dots', 'num_hyphens', 'num_at_symbols', 'num_percents', 'num_numeric_chars', 'subdomain_depth', 'path_length', 'path_depth', 'query_length', 'has_ip_address', 'uses_https', 'relative_form_action', 'submit_email', 'missing_title', 'external_favicon', 'has_iframe', 'ratio_external_links', 'ratio_null_links', 'external_css_ratio', 'has_images', 'popup_window']


## 10. Save outputs


In [ ]:
OUTDIR = './cleaned'
os.makedirs(OUTDIR, exist_ok=True)
df_A_scaled.to_csv(os.path.join(OUTDIR,'A_full.csv'), index=False)
df_B_scaled.to_csv(os.path.join(OUTDIR,'B_full.csv'), index=False)
df_C_scaled.to_csv(os.path.join(OUTDIR,'C_full.csv'), index=False)

df_A_overlap.to_csv(os.path.join(OUTDIR,'A_common.csv'), index=False)
df_B_overlap.to_csv(os.path.join(OUTDIR,'B_common.csv'), index=False)
df_C_overlap.to_csv(os.path.join(OUTDIR,'C_common.csv'), index=False)
print('Saved to', OUTDIR)


Saved to ./cleaned
